# Market-behavior representation exploration

## Provisional conventional feature bank

This notebook builds editable, horizon-based behavioral descriptors for the established study panel, inspects their distributions and pairwise relationships, and prepares a shared-column scaled matrix. The formulas are exploratory trial definitions, not final scientific claims or economic conclusions.

[Notebook 01](01_eda_and_study_panel_selection.ipynb) establishes the population and selection rationale. See the [Milestone 1 plan](../../PLAN_MILESTONE_1.md) for the broader direction.


### Conventions

- One row is an item-date on the applicable daily calendar. Absent applicable dates carry zero activity and historical order count; prices remain missing. Observed nulls and zeros are not rewritten.
- VWAP is the source average field. Historical order count is not order-book depth or trader count.
- Price paths require adjacent usable VWAP observations. A calendar gap is never treated as a daily return.
- Windows use calendar days and require complete usable inputs. The activity trials compare a recent h-day total with the preceding h-day total; an unavailable or zero reference total leaves the descriptor unavailable.
- Preparation uses complete feature rows and one standard scaling per feature column.

In [ ]:
from datetime import date

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler

from transform.notebook_access import open_analysis_connection

pio.renderers.default = "notebook"

HISTORY_START = date(2022, 6, 1)
HISTORY_END = date(2026, 9, 1)
HORIZONS = (7, 14, 30, 60)
DEFAULT_VISIBLE_HORIZONS = (7, 14, 30, 60)
HORIZON_COLORS = {
    3: "#0072B2", 5: "#E69F00", 7: "#009E73", 14: "#D55E00",
    21: "#CC79A7", 30: "#56B4E9", 60: "#332288",
}
HISTOGRAM_BINS = 60
DISTRIBUTION_FIGURE_WIDTH_PX = 1100
DISTRIBUTION_FIGURE_HEIGHT_PX = 1500
CORRELATION_CELL_SIZE_PX = 14
CORRELATION_LEFT_MARGIN_PX = 260
CORRELATION_BOTTOM_MARGIN_PX = 220

def assert_compact_figure(figure: go.Figure, budget_bytes: int = 5_000_000) -> int:
    payload_bytes = len(figure.to_json().encode())
    if payload_bytes > budget_bytes:
        raise ValueError(f"Figure JSON is {payload_bytes:,} bytes; budget is {budget_bytes:,} bytes.")
    return payload_bytes


SHORT_DESCRIPTORS = {
    "Price log change": "Price change", "Path variability": "Variability",
    "Path reversal": "Reversal", "Proportional volume activity": "Volume ratio",
    "Proportional historical order-count activity": "Order ratio",
    "Intraday high-low dispersion": "High–low range",
    "Maximum upward excursion": "Upward excursion", "Maximum downward excursion": "Downward excursion",
    "Early-to-late median shift": "Median shift", "Terminal settling": "Terminal settling",
}


def short_feature(column: str) -> str:
    horizon = column.rsplit("_", 1)[-1]
    return f"{SHORT_DESCRIPTORS[FEATURE_DESCRIPTORS[column]]} · {horizon}"

In [ ]:
INPUT_SQL = """
select
    type_id, region_id, type_name, study_category, market_date,
    average as vwap, highest, lowest, volume, order_count, estimated_traded_isk
from main.int_study_panel_history
order by type_id, market_date
"""

with open_analysis_connection() as connection:
    panel_history = pl.DataFrame(connection.execute(INPUT_SQL).to_arrow_table())

panel_history = panel_history.filter(pl.col("market_date").is_between(HISTORY_START, HISTORY_END)).with_columns(
    pl.col("market_date").cast(pl.Date, strict=True)
)
if panel_history.is_empty():
    raise ValueError("The configured study period has no history.")
if panel_history.select(pl.any_horizontal(pl.col("type_id").is_null(), pl.col("market_date").is_null()).any()).item():
    raise ValueError("Expected nonmissing item IDs and dates.")
if panel_history.select(pl.struct("type_id", "market_date").is_duplicated().any()).item():
    raise ValueError("Expected one observed history row per item-date.")

## Feature bank

The formulas below are editable trial descriptors. For each horizon h: price movement is the h-day sum of valid adjacent-day log changes; path variability is their sample standard deviation; reversal is total absolute movement minus absolute net movement; proportional volume and historical-order-count activity are recent h-day totals divided by the preceding h-day totals; intraday dispersion is the h-day mean of (highest - lowest) / VWAP.

Each trailing h-return price path contains h+1 consecutive positive finite daily VWAPs P[0] through P[h]. Maximum upward excursion is max(log(P[i]/P[0])) and maximum downward excursion is max(log(P[0]/P[i])), including i=0. Early-to-late median shift is log(median(late prices)/median(early prices)): the chronological non-overlapping early half has floor((h+1)/2) prices and the late half has the remainder. Terminal settling is the mean absolute daily log return over the final ceil(h/3) returns; lower means more settled. All four require the entire valid price path.

In [ ]:
def build_applicable_history(history: pl.DataFrame) -> pl.DataFrame:
    """Materialize the established item-by-day calendar without filling observed values."""
    item_metadata = history.select("type_id", "region_id", "type_name", "study_category").unique(
        "type_id", maintain_order=True
    )
    calendar = pl.DataFrame(
        {"market_date": pl.date_range(HISTORY_START, HISTORY_END, "1d", eager=True)}
    )
    applicable = calendar.join(item_metadata, how="cross")
    source_values = history.drop("region_id", "type_name", "study_category").with_columns(
        pl.lit(True).alias("is_observed")
    )
    return (
        applicable.join(source_values, on=["type_id", "market_date"], how="left")
        .with_columns(pl.col("is_observed").fill_null(False))
        .with_columns(
            pl.when(~pl.col("is_observed")).then(pl.lit(0.0)).otherwise(pl.col("volume")).alias("volume"),
            pl.when(~pl.col("is_observed")).then(pl.lit(0.0)).otherwise(pl.col("order_count")).alias("order_count"),
        )
        .sort("type_id", "market_date")
    )


def _finite_nonnegative(column: str) -> pl.Expr:
    return pl.col(column).is_finite() & (pl.col(column) >= 0)


def _finite_positive(column: str) -> pl.Expr:
    return pl.col(column).is_finite() & (pl.col(column) > 0)


applicable_history = build_applicable_history(panel_history)

In [ ]:
def build_feature_bank(history: pl.DataFrame, horizons: tuple[int, ...]) -> tuple[pl.DataFrame, list[str], dict[str, str]]:
    """Build provisional full-window features; unavailable price paths stay null."""
    rows = history.with_columns(
        pl.col("vwap").shift(1).over("type_id").alias("previous_vwap")
    ).with_columns(
        pl.when(_finite_positive("vwap") & _finite_positive("previous_vwap"))
        .then(pl.col("vwap").log() - pl.col("previous_vwap").log())
        .otherwise(None).alias("daily_log_return"),
        pl.when(
            _finite_positive("vwap") & pl.col("highest").is_finite()
            & pl.col("lowest").is_finite() & (pl.col("lowest") >= 0)
            & (pl.col("highest") >= pl.col("lowest"))
        )
        .then((pl.col("highest") - pl.col("lowest")) / pl.col("vwap"))
        .otherwise(None).alias("daily_relative_range"),
    )

    feature_columns, descriptors, expressions = [], {}, []
    for horizon in horizons:
        return_count = pl.col("daily_log_return").is_not_null().cast(pl.Int64).rolling_sum(horizon).over("type_id")
        range_count = pl.col("daily_relative_range").is_not_null().cast(pl.Int64).rolling_sum(horizon).over("type_id")
        net_return = pl.col("daily_log_return").rolling_sum(horizon).over("type_id")
        absolute_return = pl.col("daily_log_return").abs().rolling_sum(horizon).over("type_id")
        volume_total = pl.col("volume").rolling_sum(horizon).over("type_id")
        prior_volume_total = pl.col("volume").rolling_sum(horizon).shift(horizon).over("type_id")
        volume_valid = _finite_nonnegative("volume").cast(pl.Int64).rolling_sum(horizon).over("type_id")
        prior_volume_valid = _finite_nonnegative("volume").cast(pl.Int64).rolling_sum(horizon).shift(horizon).over("type_id")
        order_total = pl.col("order_count").rolling_sum(horizon).over("type_id")
        prior_order_total = pl.col("order_count").rolling_sum(horizon).shift(horizon).over("type_id")
        order_valid = _finite_nonnegative("order_count").cast(pl.Int64).rolling_sum(horizon).over("type_id")
        prior_order_valid = _finite_nonnegative("order_count").cast(pl.Int64).rolling_sum(horizon).shift(horizon).over("type_id")
        early_size = (horizon + 1) // 2
        late_size = horizon + 1 - early_size
        terminal_size = (horizon + 2) // 3
        log_price = pl.col("vwap").log()
        initial_log_price = log_price.shift(horizon).over("type_id")
        early_median = pl.col("vwap").rolling_median(early_size).shift(late_size).over("type_id")
        late_median = pl.col("vwap").rolling_median(late_size).over("type_id")
        definitions = {
            f"price_log_change_{horizon}d": ("Price log change", return_count, net_return),
            f"path_return_std_{horizon}d": ("Path variability", return_count, pl.col("daily_log_return").rolling_std(horizon).over("type_id")),
            f"path_reversal_{horizon}d": ("Path reversal", return_count, absolute_return - net_return.abs()),
            f"upward_excursion_{horizon}d": ("Maximum upward excursion", return_count, log_price.rolling_max(horizon + 1).over("type_id") - initial_log_price),
            f"downward_excursion_{horizon}d": ("Maximum downward excursion", return_count, initial_log_price - log_price.rolling_min(horizon + 1).over("type_id")),
            f"median_shift_{horizon}d": ("Early-to-late median shift", return_count, late_median.log() - early_median.log()),
            f"terminal_settling_{horizon}d": ("Terminal settling", return_count, pl.col("daily_log_return").abs().rolling_mean(terminal_size).over("type_id")),
            f"volume_ratio_{horizon}d": ("Proportional volume activity", volume_valid, volume_total / prior_volume_total, prior_volume_valid, prior_volume_total),
            f"order_count_ratio_{horizon}d": ("Proportional historical order-count activity", order_valid, order_total / prior_order_total, prior_order_valid, prior_order_total),
            f"mean_relative_range_{horizon}d": ("Intraday high-low dispersion", range_count, pl.col("daily_relative_range").rolling_mean(horizon).over("type_id")),
        }
        for column, definition in definitions.items():
            descriptor, current_valid, expression, *reference = definition
            available = current_valid == horizon
            if reference:
                reference_valid, reference_total = reference
                available = available & (reference_valid == horizon) & (reference_total > 0)
            expressions.append(pl.when(available).then(expression).otherwise(None).alias(column))
            feature_columns.append(column)
            descriptors[column] = descriptor

    return rows.with_columns(expressions), feature_columns, descriptors


behavior_rows, FEATURE_COLUMNS, FEATURE_DESCRIPTORS = build_feature_bank(applicable_history, HORIZONS)

## Feature distributions

Each descriptor uses common bins across its horizon overlays. Values are aggregated into exact percentage histograms before rendering; all finite observations contribute to their trace.

In [ ]:
descriptor_order = list(dict.fromkeys(FEATURE_DESCRIPTORS.values()))
distribution_histograms = {}
for descriptor in descriptor_order:
    columns = [column for column in FEATURE_COLUMNS if FEATURE_DESCRIPTORS[column] == descriptor]
    values = [behavior_rows.get_column(column).cast(pl.Float64).to_numpy() for column in columns]
    values = [value[np.isfinite(value)] for value in values]
    combined = np.concatenate([value for value in values if len(value)])
    start, end = float(combined.min()), float(combined.max())
    if start == end:
        start, end = start - 0.5, end + 0.5
    edges = np.linspace(start, end, HISTOGRAM_BINS + 1)
    counts = [np.histogram(value, bins=edges)[0] for value in values]
    percentages = [np.r_[count * 100 / len(value), 0.0] for count, value in zip(counts, values, strict=True)]
    distribution_histograms[descriptor] = (columns, edges, percentages)

In [ ]:
def plot_feature_distributions(histograms: dict) -> go.Figure:
    figure = make_subplots(rows=5, cols=2, subplot_titles=[SHORT_DESCRIPTORS[d] for d in descriptor_order])
    for descriptor_index, descriptor in enumerate(descriptor_order):
        row, col = divmod(descriptor_index, 2)
        columns, edges, percentages = histograms[descriptor]
        for column, percent in zip(columns, percentages, strict=True):
            horizon = int(column.rsplit("_", 1)[-1][:-1])
            figure.add_trace(
                go.Scatter(
                    x=edges, y=percent, mode="lines",
                    line={"shape": "hv", "color": HORIZON_COLORS[horizon]},
                    name=f"{horizon} days", legendgroup=f"horizon-{horizon}",
                    showlegend=descriptor_index == 0,
                    visible=True if horizon in DEFAULT_VISIBLE_HORIZONS else "legendonly",
                    hovertemplate=f"{descriptor}<br>{horizon} days<br>bin start=%{{x}}<br>percent=%{{y:.3f}}<extra></extra>",
                ), row=row + 1, col=col + 1,
            )
        figure.update_xaxes(showgrid=False, row=row + 1, col=col + 1)
        figure.update_yaxes(title_text="Observations (%)", row=row + 1, col=col + 1)
    figure.update_layout(
        title="Feature distributions by horizon", width=DISTRIBUTION_FIGURE_WIDTH_PX,
        height=DISTRIBUTION_FIGURE_HEIGHT_PX, template="plotly_white",
        legend={"groupclick": "togglegroup", "orientation": "h", "y": 1.08, "x": 0},
        margin={"l": 75, "r": 30, "t": 130, "b": 50},
        font={"size": 12, "color": "#334155"},
    )
    return figure

In [ ]:
distribution_figure = plot_feature_distributions(distribution_histograms)
distribution_payload_bytes = assert_compact_figure(distribution_figure)
distribution_figure

## Feature correlations

Pairwise Pearson correlations use all finite observation pairs for each feature combination. Average-linkage clustering orders features using dissimilarity 1 − |r|, bringing strong positive and negative relationships together. The same order is used on both axes; reordering does not change the signed correlations shown or establish a pruning threshold.

In [ ]:
values = behavior_rows.select(pl.col(column).cast(pl.Float64) for column in FEATURE_COLUMNS).to_numpy()
correlations = np.full((len(FEATURE_COLUMNS), len(FEATURE_COLUMNS)), np.nan)
counts = np.zeros((len(FEATURE_COLUMNS), len(FEATURE_COLUMNS)), dtype=int)
for left in range(len(FEATURE_COLUMNS)):
    for right in range(len(FEATURE_COLUMNS)):
        valid = np.isfinite(values[:, left]) & np.isfinite(values[:, right])
        counts[left, right] = valid.sum()
        if valid.sum() >= 2:
            pair = values[valid][:, [left, right]]
            if np.ptp(pair[:, 0]) and np.ptp(pair[:, 1]):
                correlations[left, right] = np.corrcoef(pair[:, 0], pair[:, 1])[0, 1]

In [ ]:
def plot_feature_correlations(correlations: np.ndarray, counts: np.ndarray) -> go.Figure:
    from scipy.cluster.hierarchy import leaves_list, linkage
    from scipy.spatial.distance import squareform

    feature_count = len(FEATURE_COLUMNS)
    expected_shape = (feature_count, feature_count)
    if feature_count < 2 or len(set(FEATURE_COLUMNS)) != feature_count:
        raise ValueError("Clustering requires at least two uniquely named features")
    if correlations.shape != expected_shape or counts.shape != expected_shape:
        raise ValueError("Correlation and count matrices must match the feature bank")
    if not np.isfinite(correlations).all():
        raise ValueError("Clustering requires finite pairwise correlations")
    if not np.allclose(correlations, correlations.T, atol=1e-10, rtol=0):
        raise ValueError("Correlation matrix must be symmetric")
    if not np.allclose(np.diag(correlations), 1, atol=1e-10, rtol=0):
        raise ValueError("Correlation diagonal must equal one")
    if np.any(np.abs(correlations) > 1 + 1e-10):
        raise ValueError("Correlations must be between -1 and 1")
    if not np.isfinite(counts).all() or np.any(counts < 0) or not np.array_equal(counts, counts.T):
        raise ValueError("Pair counts must be finite, nonnegative, and symmetric")

    # Fixed feature input order makes tie handling reproducible. Symmetrize and
    # clip only the clustering distances to absorb floating-point roundoff.
    distances = 1 - np.clip(np.abs(correlations), 0, 1)
    distances = (distances + distances.T) / 2
    np.fill_diagonal(distances, 0)
    order = leaves_list(linkage(squareform(distances), method="average"))
    ordered_features = [FEATURE_COLUMNS[index] for index in order]
    ordered_correlations = correlations[np.ix_(order, order)]
    ordered_counts = counts[np.ix_(order, order)]

    grid_size = feature_count * CORRELATION_CELL_SIZE_PX
    figure = go.Figure(go.Heatmap(
        x=ordered_features, y=ordered_features, z=ordered_correlations, customdata=ordered_counts,
        text=[[f"{FEATURE_DESCRIPTORS[left]} ({left})<br>{FEATURE_DESCRIPTORS[right]} ({right})" for right in ordered_features] for left in ordered_features],
        zmin=-1, zmax=1, zmid=0, colorscale="RdBu_r", colorbar={"title": "Pearson r"},
        hovertemplate="%{text}<br>r=%{z:.3f}<br>finite pairs=%{customdata:,}<extra></extra>",
    ))
    figure.update_layout(
        title="Feature correlations (clustered by |r|)",
        width=CORRELATION_LEFT_MARGIN_PX + grid_size + 120,
        height=CORRELATION_BOTTOM_MARGIN_PX + grid_size + 90,
        margin={"l": CORRELATION_LEFT_MARGIN_PX, "b": CORRELATION_BOTTOM_MARGIN_PX, "r": 120, "t": 70},
        template="plotly_white",
    )
    figure.update_xaxes(tickmode="array", tickvals=ordered_features, ticktext=[short_feature(c) for c in ordered_features], tickangle=-55, tickfont={"size": 9}, constrain="domain", categoryorder="array", categoryarray=ordered_features)
    figure.update_yaxes(tickmode="array", tickvals=ordered_features, ticktext=[short_feature(c) for c in ordered_features], tickfont={"size": 9}, autorange="reversed", scaleanchor="x", scaleratio=1, categoryorder="array", categoryarray=ordered_features)
    return figure

In [ ]:
correlation_figure = plot_feature_correlations(correlations, counts)
correlation_payload_bytes = assert_compact_figure(correlation_figure)
correlation_figure

## Prepared and scaled matrix

Rows enter only when every provisional feature is finite. This complete-case matrix is prepared for later exploration.

In [ ]:
finite_feature_row = pl.all_horizontal([pl.col(column).is_finite() for column in FEATURE_COLUMNS]).fill_null(False)
prepared_rows = behavior_rows.filter(finite_feature_row)
if prepared_rows.is_empty():
    raise ValueError("No rows have every provisional feature available.")

X_raw = prepared_rows.select(pl.col(column).cast(pl.Float64) for column in FEATURE_COLUMNS).to_numpy()
if np.any(np.std(X_raw, axis=0) == 0):
    constant_columns = [column for index, column in enumerate(FEATURE_COLUMNS) if np.std(X_raw[:, index]) == 0]
    raise ValueError(f"Constant provisional feature columns: {constant_columns}")

feature_scaler = StandardScaler()
X_scaled = feature_scaler.fit_transform(X_raw)
prepared_metadata = prepared_rows.select("type_id", "type_name", "study_category", "market_date")

## Illustrative case signatures

This reproducible record is an exploratory illustration from the prepared feature matrix, not a final event label or validated method.

In [ ]:
from datetime import timedelta

SIGNATURE_HORIZONS = (7, 14, 30, 60)
CASE_DISPLAY_DAYS_BEFORE = 90
CASE_DISPLAY_DAYS_AFTER = 30
DESCRIPTOR_PREFIXES = {"Price log change": "price_log_change", "Path variability": "path_return_std", "Path reversal": "path_reversal", "Maximum upward excursion": "upward_excursion", "Maximum downward excursion": "downward_excursion", "Early-to-late median shift": "median_shift", "Terminal settling": "terminal_settling", "Proportional volume activity": "volume_ratio", "Proportional historical order-count activity": "order_count_ratio", "Intraday high-low dispersion": "mean_relative_range"}

# Edit this shared definition to replace or add illustrative records.
CASES = (
    {"case_id": "lowest-standardized-signature-distance", "label": "Illustrative standardized signature", "short_label": "Illustrative signature", "type_id": 17938, "endpoint": date(2026, 2, 10), "display_range": (-CASE_DISPLAY_DAYS_BEFORE, CASE_DISPLAY_DAYS_AFTER), "color": "#009E73", "selection_rationale": "Illustrative prepared signature retained from the earlier exploration."},
)

In [ ]:
case_definition_rows = pl.DataFrame(CASES).with_row_index("case_order")
if case_definition_rows["case_id"].n_unique() != case_definition_rows.height:
    raise ValueError("Case IDs must be unique.")
prepared_keyed_rows = prepared_metadata.with_row_index("matrix_row")
case_key_matches = (
    case_definition_rows.join(
        prepared_keyed_rows,
        left_on=["type_id", "endpoint"],
        right_on=["type_id", "market_date"],
        how="left",
    )
    .sort("case_order")
)
missing_case_signatures = case_key_matches.filter(pl.col("matrix_row").is_null())
if missing_case_signatures.height:
    raise ValueError(f"Unavailable exact signature rows: {missing_case_signatures['case_id'].to_list()}")
if case_key_matches.select(["type_id", "endpoint"]).n_unique() != case_key_matches.height:
    raise ValueError("Each case must identify one exact type_id + endpoint signature row.")

CASE_SIGNATURES = {
    row["case_id"]: {
        "raw": X_raw[int(row["matrix_row"])],
        "standardized": X_scaled[int(row["matrix_row"])],
        "matrix_row": int(row["matrix_row"]),
    }
    for row in case_key_matches.to_dicts()
}

CASE_PLOT_DATA = {}
for case in CASES:
    signature = CASE_SIGNATURES[case["case_id"]]
    start_offset, end_offset = case["display_range"]
    source = behavior_rows.filter(
        (pl.col("type_id") == case["type_id"])
        & pl.col("market_date").is_between(
            case["endpoint"] + timedelta(days=start_offset),
            case["endpoint"] + timedelta(days=end_offset),
        )
    ).sort("market_date")
    if source.is_empty():
        raise ValueError(f"No source history for {case['case_id']}")
    signature_by_descriptor = {}
    for descriptor, prefix in DESCRIPTOR_PREFIXES.items():
        indices = [FEATURE_COLUMNS.index(f"{prefix}_{horizon}d") for horizon in SIGNATURE_HORIZONS]
        signature_by_descriptor[descriptor] = {
            "raw": signature["raw"][indices],
            "standardized": signature["standardized"][indices],
        }
    CASE_PLOT_DATA[case["case_id"]] = {"case": case, "signature": signature_by_descriptor, "source": source}


In [ ]:
def plot_case_signature(prepared: dict[str, object]) -> go.Figure:
    case = prepared["case"]
    signature = prepared["signature"]
    source = prepared["source"]
    figure = make_subplots(
        rows=5, cols=4,
        specs=[[{"colspan": 2, "rowspan": 3}, None, {}, {}], [None, None, {}, {}], [None, None, {}, {}], [{"colspan": 2, "rowspan": 2}, None, {}, {}], [None, None, {}, {}]],
        subplot_titles=[
            "VWAP",
            *[SHORT_DESCRIPTORS[d] for d in list(DESCRIPTOR_PREFIXES)[:6]],
            "Volume",
            *[SHORT_DESCRIPTORS[d] for d in list(DESCRIPTOR_PREFIXES)[6:]],
        ],
        column_widths=[0.24, 0.24, 0.26, 0.26],
        horizontal_spacing=0.075, vertical_spacing=0.08,
    )
    figure.add_trace(go.Scatter(x=source["market_date"].to_list(), y=source["vwap"].to_list(), name="VWAP", showlegend=False, line={"color": case["color"]}), row=1, col=1)
    figure.add_trace(go.Bar(x=source["market_date"].to_list(), y=source["volume"].to_list(), name="Volume", showlegend=False, marker_color=case["color"], opacity=0.65), row=4, col=1)
    figure.update_xaxes(matches="x", row=4, col=1)
    for row in (1, 4):
        figure.add_vline(x=case["endpoint"], line_dash="dash", line_color=case["color"], row=row, col=1)
    for index, (descriptor, prefix) in enumerate(DESCRIPTOR_PREFIXES.items()):
        row, col = divmod(index, 2)
        for values, representation, visible in ((signature[descriptor]["raw"], "Raw", True), (signature[descriptor]["standardized"], "Standardized", False)):
            figure.add_trace(go.Bar(
                x=[f"{horizon}d" for horizon in SIGNATURE_HORIZONS], y=values,
                name=representation, visible=visible, showlegend=False,
                marker_color=[HORIZON_COLORS[horizon] for horizon in SIGNATURE_HORIZONS],
                hovertemplate=f"{descriptor}<br>%{{x}}<br>{representation.lower()}=%{{y:.4g}}<extra></extra>",
            ), row=row + 1, col=col + 3)
    source_trace_count = 2
    signature_trace_count = len(figure.data) - source_trace_count
    raw_visibility = [True, True] + [index % 2 == 0 for index in range(signature_trace_count)]
    standardized_visibility = [True, True] + [index % 2 == 1 for index in range(signature_trace_count)]
    assert all(raw_visibility[:2]) and all(standardized_visibility[:2])
    figure.update_layout(
        title=f"{case['label']} — {source['type_name'][0]} (type {case['type_id']}) through {case['endpoint']}",
        template="plotly_white", barmode="group", width=1100, height=1400,
        margin={"l": 75, "r": 30, "t": 145, "b": 55},
        font={"size": 12, "color": "#334155"},
        updatemenus=[{"type": "dropdown", "x": 1, "xanchor": "right", "y": 1.08, "buttons": [
            {"label": "Raw signature", "method": "update", "args": [{"visible": raw_visibility}]},
            {"label": "Standardized signature", "method": "update", "args": [{"visible": standardized_visibility}]},
        ]}],
    )
    figure.update_yaxes(title_text="ISK", row=1, col=1)
    figure.update_yaxes(title_text="Units", row=4, col=1)
    assert_compact_figure(figure)
    return figure

In [ ]:
case_signature_figure_lowest_standardized_signature_distance = plot_case_signature(CASE_PLOT_DATA[CASES[0]["case_id"]])
case_signature_figure_lowest_standardized_signature_distance

## Retrospective PCA inspection

PCA is fit to the shared-column standardized prepared matrix for exploratory inspection only. Coefficients below are PCA component coefficients, not factor interpretations or a retained-dimensionality decision.

In [ ]:
from sklearn.decomposition import PCA

PCA_COMPONENT_DISPLAY_LIMIT = 8
pca_model = PCA()
pca_scores_array = pca_model.fit_transform(X_scaled)
pca_score_keys = prepared_metadata.select("type_id", "type_name", "study_category", "market_date")
pca_scores = pca_score_keys.with_columns(
    [pl.Series(f"pc_{component + 1}", pca_scores_array[:, component]) for component in range(pca_scores_array.shape[1])]
)
pca_coefficients = pca_model.components_
assert pca_scores.select(["type_id", "market_date"]).n_unique() == pca_scores.height
assert pca_scores.height == X_scaled.shape[0]
assert np.isclose(pca_model.explained_variance_ratio_.sum(), 1.0)
assert np.allclose(pca_model.inverse_transform(pca_scores_array), X_scaled, rtol=1e-10, atol=1e-10)

### Explained variance

In [ ]:
component_numbers = np.arange(1, len(pca_model.explained_variance_ratio_) + 1)
individual = pca_model.explained_variance_ratio_
cumulative = np.cumsum(individual)

In [ ]:
def plot_pca_explained_variance(component_numbers, individual, cumulative) -> go.Figure:
    figure = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Individual", "Cumulative"],
        horizontal_spacing=0.12,
    )
    figure.add_trace(
        go.Bar(
            x=component_numbers,
            y=individual,
            name="Individual explained variance ratio",
            marker_color="#0072B2",
            hovertemplate="PC%{x}<br>ratio=%{y:.3%}<extra></extra>",
        ),
        row=1,
        col=1,
    )
    figure.add_trace(
        go.Scatter(
            x=component_numbers,
            y=cumulative,
            mode="lines+markers",
            name="Cumulative explained variance ratio",
            line={"color": "#0072B2"},
            hovertemplate="PC%{x}<br>cumulative=%{y:.3%}<extra></extra>",
        ),
        row=1,
        col=2,
    )
    figure.update_layout(title="PCA explained variance across all components", template="plotly_white", width=1100, height=420, showlegend=False, font={"size": 12, "color": "#334155"})
    figure.update_xaxes(title_text="Principal component")
    figure.update_yaxes(title_text="Explained variance ratio", tickformat=".0%", row=1, col=1)
    figure.update_yaxes(title_text="Cumulative variance", tickformat=".0%", range=[0, 1.02], row=1, col=2)
    assert_compact_figure(figure)
    return figure


In [ ]:
pca_explained_variance_figure = plot_pca_explained_variance(component_numbers, individual, cumulative)
pca_explained_variance_figure

### Component coefficients

In [ ]:
component_count = min(PCA_COMPONENT_DISPLAY_LIMIT, pca_coefficients.shape[0])
component_labels = [f"PC {component}" for component in range(1, component_count + 1)]
coefficient_matrix = pca_coefficients[:component_count].T
feature_columns = FEATURE_COLUMNS
hover_feature_names = np.repeat(
    np.array([f"{FEATURE_DESCRIPTORS[column]} ({column})" for column in feature_columns])[:, None],
    component_count,
    axis=1,
)
color_limit = float(np.abs(coefficient_matrix).max())
feature_ticktext = [short_feature(c) for c in feature_columns]
figure_height = max(650, 18 * len(feature_columns) + 180)

In [ ]:
def plot_pca_coefficient_heatmap(
    coefficient_matrix, component_labels, feature_columns, hover_feature_names, color_limit, feature_ticktext, figure_height,
) -> go.Figure:
    figure = go.Figure(
        go.Heatmap(
            z=coefficient_matrix,
            x=component_labels,
            y=feature_columns,
            customdata=hover_feature_names,
            colorscale="RdBu_r",
            zmid=0,
            zmin=-color_limit,
            zmax=color_limit,
            colorbar={"title": "PCA coefficient"},
            hovertemplate="%{customdata}<br>%{x}<br>coefficient=%{z:.4f}<extra></extra>",
        )
    )
    figure.update_layout(
        title="PCA component coefficients by shared feature column",
        template="plotly_white",
        width=1100,
        height=figure_height,
        xaxis_title="Principal component",
        yaxis_title="Feature",
        font={"size": 12, "color": "#334155"},
    )
    figure.update_yaxes(tickmode="array", tickvals=feature_columns, ticktext=feature_ticktext, tickfont={"size": 10}, autorange="reversed")
    assert_compact_figure(figure)
    return figure


In [ ]:
pca_coefficient_heatmap = plot_pca_coefficient_heatmap(
    coefficient_matrix, component_labels, feature_columns, hover_feature_names, color_limit, feature_ticktext, figure_height
)
pca_coefficient_heatmap

## All-row 2D PCA viewer

The view retains every prepared row. Full range shows the complete extent; central range zooms each axis to its 1st–99th percentiles without removing data. Density and hexbin calculations always use the full population. The density mode is a binned Gaussian kernel-density estimate: exact 2D histogram counts are convolved with a Gaussian kernel whose bandwidth is specified in grid cells, then normalized by retained-row count and cell area. It is not a count heatmap. Plotly has no native Cartesian hexbin trace, so the hexbin mode uses exact hexagon assignments and hoverable filled polygon traces; its counts sum to the retained population. Dropdowns switch only precomputed browser-side traces.

In [ ]:
# Editable display, density, and grid settings. These remain bounded to keep browser controls responsive.
PCA_VIEWER_WIDTH_PX = 1_000
PCA_VIEWER_HEIGHT_PX = 750
PCA_SCATTER_OPACITY = 0.20
PCA_SCATTER_MARKER_SIZE = 3
PCA_HEXBIN_COLUMNS = 34
PCA_HEXBIN_OPACITY = 0.78
PCA_DENSITY_GRID_SIZE = 80
PCA_DENSITY_BANDWIDTH_CELLS = 1.6
PCA_VIEWER_PAYLOAD_BUDGET_BYTES = 24_000_000
PCA_VIEWER_TOTAL_PAYLOAD_BUDGET_BYTES = 45_000_000

PCA_RETAINED_ROW_COUNT = pca_scores.height
if PCA_RETAINED_ROW_COUNT <= 0:
    raise ValueError("Expected at least one retained PCA row.")
if pca_scores.height != prepared_metadata.height or pca_scores_array.shape[0] != PCA_RETAINED_ROW_COUNT:
    raise ValueError("PCA scores must remain keyed one-to-one with prepared metadata.")

PCA_TYPE_IDS = prepared_metadata.get_column("type_id").to_numpy()
PCA_TYPE_NAMES = prepared_metadata.get_column("type_name").to_numpy()
PCA_DATES_ISO = prepared_metadata.get_column("market_date").to_numpy().astype("datetime64[D]").astype(str)
PCA_SCORES = np.asarray(pca_scores_array, dtype=np.float64)
PCA_UNIQUE_TYPE_IDS, PCA_TYPE_INVERSES = np.unique(PCA_TYPE_IDS, return_inverse=True)
PCA_TYPE_ROW_INDICES = [np.flatnonzero(PCA_TYPE_INVERSES == type_index) for type_index in range(PCA_UNIQUE_TYPE_IDS.size)]
PCA_TYPE_LABELS = [str(PCA_TYPE_NAMES[rows[0]]) for rows in PCA_TYPE_ROW_INDICES]

PCA_CASE_ROWS = case_key_matches.select("case_id", "label", "short_label", "color", "type_id", "type_name", "endpoint", "matrix_row")
if PCA_CASE_ROWS.height != len(CASES):
    raise ValueError("Every configured case must have one exact prepared PCA row.")
PCA_CASE_MATRIX_ROWS = PCA_CASE_ROWS.get_column("matrix_row").to_numpy().astype(int)
if not np.array_equal(PCA_SCORES[PCA_CASE_MATRIX_ROWS], pca_scores_array[PCA_CASE_MATRIX_ROWS]):
    raise ValueError("Keyed case coordinates do not exactly match PCA score rows.")


def _pca_axis_title(component: int) -> str:
    return f"PC{component + 1}"


def _pca_central_range(values: np.ndarray) -> list[float]:
    return np.quantile(values, [0.01, 0.99]).tolist()


def _pca_ranges(x: np.ndarray, y: np.ndarray) -> tuple[list[float], list[float]]:
    def padded(values: np.ndarray) -> list[float]:
        lower, upper = float(values.min()), float(values.max())
        padding = max((upper - lower) * 0.04, 0.05)
        return [lower - padding, upper + padding]
    return padded(x), padded(y)


def _round_pointy_axial(q_float: np.ndarray, r_float: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    cube_x, cube_z = q_float, r_float
    cube_y = -cube_x - cube_z
    rounded_x, rounded_y, rounded_z = np.rint(cube_x), np.rint(cube_y), np.rint(cube_z)
    x_delta, y_delta, z_delta = np.abs(rounded_x - cube_x), np.abs(rounded_y - cube_y), np.abs(rounded_z - cube_z)
    replace_x = (x_delta > y_delta) & (x_delta > z_delta)
    replace_y = ~replace_x & (y_delta > z_delta)
    rounded_x[replace_x] = -rounded_y[replace_x] - rounded_z[replace_x]
    rounded_y[replace_y] = -rounded_x[replace_y] - rounded_z[replace_y]
    return rounded_x.astype(np.int64), rounded_z.astype(np.int64)


def _pointy_hex_vertices(center_x: float, center_y: float, radius: float) -> tuple[np.ndarray, np.ndarray]:
    angles = np.deg2rad(np.arange(30, 391, 60))
    return center_x + radius * np.cos(angles), center_y + radius * np.sin(angles)


def _hexbin(x: np.ndarray, y: np.ndarray, x_range: list[float], columns: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    radius = (x_range[1] - x_range[0]) / (columns * np.sqrt(3))
    dx, dy = x - x_range[0], y - y.min()
    q_float = np.sqrt(3) * dx / (3 * radius) - dy / (3 * radius)
    r_float = 2 * dy / (3 * radius)
    q, r = _round_pointy_axial(q_float, r_float)
    keys = q * 1_000_000 + r + 500_000
    unique_keys, _, counts = np.unique(keys, return_inverse=True, return_counts=True)
    q = unique_keys // 1_000_000
    r = unique_keys % 1_000_000 - 500_000
    centers_x = x_range[0] + radius * np.sqrt(3) * (q + r / 2)
    centers_y = y.min() + radius * 1.5 * r
    if int(counts.sum()) != len(x):
        raise ValueError("Hexbin counts must conserve the complete PCA population.")
    return centers_x, centers_y, counts, radius


def _validate_pointy_hex_geometry() -> None:
    radius = 1.0
    vertices_x, vertices_y = _pointy_hex_vertices(0.0, 0.0, radius)
    if not np.allclose(np.hypot(vertices_x[:-1], vertices_y[:-1]), radius):
        raise ValueError("Pointy hexagon vertices must lie at the configured radius.")
    inside_x = np.array([0.0, 0.80, -0.80])  # Inside the +/- sqrt(3)/2 pointy-hex boundary.
    outside_x = np.array([0.94, -0.94])  # Across that same boundary into adjacent hexagons.
    inside_q, inside_r = _round_pointy_axial(np.sqrt(3) * inside_x / (3 * radius), np.zeros(3))
    outside_q, outside_r = _round_pointy_axial(np.sqrt(3) * outside_x / (3 * radius), np.zeros(2))
    if not (np.array_equal(inside_q, [0, 0, 0]) and np.array_equal(inside_r, [0, 0, 0])):
        raise ValueError("Representative inside points must assign to the center pointy hexagon.")
    if not (np.array_equal(outside_q, [1, -1]) and np.array_equal(outside_r, [0, 0])):
        raise ValueError("Representative boundary-crossing points must assign to adjacent pointy hexagons.")


_validate_pointy_hex_geometry()


def _gaussian_density(x: np.ndarray, y: np.ndarray, x_range: list[float], y_range: list[float]) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    edges_x = np.linspace(*x_range, PCA_DENSITY_GRID_SIZE + 1)
    edges_y = np.linspace(*y_range, PCA_DENSITY_GRID_SIZE + 1)
    counts, _, _ = np.histogram2d(x, y, bins=(edges_x, edges_y))
    kernel_radius = int(np.ceil(3 * PCA_DENSITY_BANDWIDTH_CELLS))
    kernel_axis = np.arange(-kernel_radius, kernel_radius + 1, dtype=float)
    kernel = np.exp(-0.5 * (kernel_axis / PCA_DENSITY_BANDWIDTH_CELLS) ** 2)
    kernel /= kernel.sum()
    smoothed = np.apply_along_axis(lambda values: np.convolve(values, kernel, mode="same"), 0, counts)
    smoothed = np.apply_along_axis(lambda values: np.convolve(values, kernel, mode="same"), 1, smoothed)
    cell_area = (edges_x[1] - edges_x[0]) * (edges_y[1] - edges_y[0])
    density = smoothed / (len(x) * cell_area)
    if not np.isfinite(density).all() or (density < 0).any():
        raise ValueError("Density grid must be finite and nonnegative.")
    return (edges_x[:-1] + edges_x[1:]) / 2, (edges_y[:-1] + edges_y[1:]) / 2, density.T





In [ ]:
PCA_2D_X = PCA_SCORES[:, 0]
PCA_2D_Y = PCA_SCORES[:, 1]
PCA_2D_X_RANGE, PCA_2D_Y_RANGE = _pca_ranges(PCA_2D_X, PCA_2D_Y)
PCA_2D_X_CENTRAL = _pca_central_range(PCA_2D_X)
PCA_2D_Y_CENTRAL = _pca_central_range(PCA_2D_Y)
PCA_2D_HEX_X, PCA_2D_HEX_Y, PCA_2D_HEX_COUNTS, PCA_2D_HEX_RADIUS = _hexbin(PCA_2D_X, PCA_2D_Y, PCA_2D_X_RANGE, PCA_HEXBIN_COLUMNS)
PCA_2D_HEX_VERTICES = [_pointy_hex_vertices(float(x), float(y), PCA_2D_HEX_RADIUS) for x, y in zip(PCA_2D_HEX_X, PCA_2D_HEX_Y, strict=True)]
PCA_2D_HEX_MIN, PCA_2D_HEX_MAX = int(PCA_2D_HEX_COUNTS.min()), int(PCA_2D_HEX_COUNTS.max())
PCA_2D_DENSITY_X, PCA_2D_DENSITY_Y, PCA_2D_DENSITY = _gaussian_density(PCA_2D_X, PCA_2D_Y, PCA_2D_X_RANGE, PCA_2D_Y_RANGE)
assert sum(len(rows) for rows in PCA_TYPE_ROW_INDICES) == PCA_RETAINED_ROW_COUNT
assert int(PCA_2D_HEX_COUNTS.sum()) == PCA_RETAINED_ROW_COUNT
assert np.isfinite(PCA_2D_DENSITY).all() and (PCA_2D_DENSITY >= 0).all()
for rows, type_id, label in zip(PCA_TYPE_ROW_INDICES, PCA_UNIQUE_TYPE_IDS, PCA_TYPE_LABELS, strict=True):
    assert np.all(PCA_TYPE_IDS[rows] == type_id) and np.all(PCA_TYPE_NAMES[rows] == label)
for case in PCA_CASE_ROWS.to_dicts():
    row = int(case['matrix_row'])
    assert case['endpoint'].isoformat() == PCA_DATES_ISO[row]
    assert int(PCA_TYPE_IDS[row]) == case['type_id']
    assert np.array_equal(PCA_SCORES[row, :2], pca_scores_array[row, :2])
PCA_2D_HEX_UNIQUE_COUNTS = np.unique(PCA_2D_HEX_COUNTS)
PCA_2D_EXPLAINED_VARIANCE = pca_model.explained_variance_ratio_[:2].copy()
PCA_2D_PREPARED = {
    "pca_unique_type_ids": PCA_UNIQUE_TYPE_IDS,
    "pca_type_labels": PCA_TYPE_LABELS,
    "pca_type_row_indices": PCA_TYPE_ROW_INDICES,
    "pca_dates_iso": PCA_DATES_ISO,
    "pca_2d_x": PCA_2D_X,
    "pca_2d_y": PCA_2D_Y,
    "pca_2d_x_range": PCA_2D_X_RANGE,
    "pca_2d_y_range": PCA_2D_Y_RANGE,
    "pca_2d_x_central": PCA_2D_X_CENTRAL,
    "pca_2d_y_central": PCA_2D_Y_CENTRAL,
    "pca_2d_hex_x": PCA_2D_HEX_X,
    "pca_2d_hex_y": PCA_2D_HEX_Y,
    "pca_2d_hex_counts": PCA_2D_HEX_COUNTS,
    "pca_2d_hex_vertices": PCA_2D_HEX_VERTICES,
    "pca_2d_hex_min": PCA_2D_HEX_MIN,
    "pca_2d_hex_max": PCA_2D_HEX_MAX,
    "pca_2d_hex_unique_counts": PCA_2D_HEX_UNIQUE_COUNTS,
    "pca_2d_density_x": PCA_2D_DENSITY_X,
    "pca_2d_density_y": PCA_2D_DENSITY_Y,
    "pca_2d_density": PCA_2D_DENSITY,
    "cases": PCA_CASE_ROWS.to_dicts(),
    "explained_variance": PCA_2D_EXPLAINED_VARIANCE,
}


In [ ]:
def plot_pca_2d(prepared):
    def _hex_fillcolor(count: int, minimum: int, maximum: int) -> str:
        fraction = 0.5 if maximum == minimum else (count - minimum) / (maximum - minimum)
        hue = 210 - 28 * fraction
        lightness = 95 - 55 * fraction
        return f"hsla({hue:.8f},75%,{lightness:.8f}%,{PCA_HEXBIN_OPACITY})"
    
    
    def _hex_colorscale(unique_counts: np.ndarray, minimum: int, maximum: int) -> list[list[float | str]]:
        if minimum == maximum:
            color = _hex_fillcolor(minimum, minimum, maximum)
            return [[0.0, color], [1.0, color]]
        return [
            [(int(count) - minimum) / (maximum - minimum), _hex_fillcolor(int(count), minimum, maximum)]
            for count in unique_counts
        ]
    
    
    if _hex_colorscale(np.array([7]), 7, 7) != [[0.0, _hex_fillcolor(7, 7, 7)], [1.0, _hex_fillcolor(7, 7, 7)]]:
        raise ValueError("Equal-count hexbin colorscales must have two constant endpoints.")
    
    
    def _pca_background_traces(x: np.ndarray, y: np.ndarray) -> list[go.Scattergl]:
        traces = []
        for type_id, label, row_indices in zip(prepared['pca_unique_type_ids'], prepared['pca_type_labels'], prepared['pca_type_row_indices'], strict=True):
            traces.append(go.Scattergl(
                x=x[row_indices], y=y[row_indices], mode="markers", name=label,
                meta=[int(type_id), label], customdata=prepared['pca_dates_iso'][row_indices],
                marker={"size": PCA_SCATTER_MARKER_SIZE, "color": "#4c78a8", "opacity": PCA_SCATTER_OPACITY},
                hovertemplate="%{meta[1]} (type %{meta[0]})<br>date=%{customdata}<br>PC1=%{x:.4f}<br>PC2=%{y:.4f}<extra></extra>",
                showlegend=False,
            ))
        return traces
    
    
    def _pca_case_traces_2d(x: np.ndarray, y: np.ndarray) -> list[go.Scattergl]:
        traces = []
        for case in prepared["cases"]:
            row = int(case["matrix_row"])
            traces.append(go.Scattergl(
                x=[x[row]], y=[y[row]], mode="markers", name=str(case["short_label"]),
                meta=[str(case["case_id"]), int(case["type_id"]), str(case["type_name"])],
                customdata=[case["endpoint"].isoformat()],
                marker={"size": 12, "color": str(case["color"]), "line": {"color": "#111111", "width": 1}},
                hovertemplate="%{meta[0]}<br>%{meta[2]} (type %{meta[1]})<br>date=%{customdata}<br>PC1=%{x:.4f}<br>PC2=%{y:.4f}<extra></extra>",
                showlegend=True,
            ))
        return traces
    
    
    def _pca_hexbin_traces() -> tuple[list[go.Scatter], go.Scatter]:
        centers_x, centers_y, counts = prepared['pca_2d_hex_x'], prepared['pca_2d_hex_y'], prepared['pca_2d_hex_counts']
        minimum, maximum = prepared['pca_2d_hex_min'], prepared['pca_2d_hex_max']
        traces = []
        for center_x, center_y, count, (vertices_x, vertices_y) in zip(centers_x, centers_y, counts, prepared['pca_2d_hex_vertices'], strict=True):
            traces.append(go.Scatter(
                x=vertices_x, y=vertices_y, mode="lines", fill="toself", hoveron="fills",
                fillcolor=_hex_fillcolor(int(count), minimum, maximum),
                line={"color": "rgba(255,255,255,0.35)", "width": 0.5},
                meta=[int(count), float(center_x), float(center_y)], showlegend=False,
                hovertemplate="hexbin<br>count=%{meta[0]:,}<br>PC1 center=%{meta[1]:.4f}<br>PC2 center=%{meta[2]:.4f}<extra></extra>",
            ))
        colorscale = _hex_colorscale(prepared['pca_2d_hex_unique_counts'], minimum, maximum)
        colorbar_trace = go.Scatter(
            x=[None, None], y=[None, None], mode="markers", hoverinfo="skip", showlegend=False,
            marker={
                "size": 0.1, "color": [minimum, maximum], "cmin": minimum, "cmax": maximum,
                "colorscale": colorscale, "showscale": True,
                "colorbar": {"title": "Rows per hexbin"},
            },
        )
        if maximum > minimum and len({trace.fillcolor for trace in traces}) <= 1:
            raise ValueError("Distinct hexbin counts must receive distinct quantitative colors.")
        return traces, colorbar_trace
    
    
    def _pca_density_trace() -> go.Contour:
        grid_x, grid_y, density = prepared['pca_2d_density_x'], prepared['pca_2d_density_y'], prepared['pca_2d_density']
        return go.Contour(
            x=grid_x, y=grid_y, z=density, colorscale="Blues", contours={"coloring": "fill", "showlines": True},
            colorbar={"title": "Estimated density"},
            hovertemplate="PC1=%{x:.4f}<br>PC2=%{y:.4f}<br>estimated density=%{z:.6g}<extra></extra>",
            name="Gaussian KDE", showlegend=False,
        )
    
    PCA_2D_SCATTER_TRACES = _pca_background_traces(prepared['pca_2d_x'], prepared['pca_2d_y'])
    PCA_2D_HEXBIN_TRACES, PCA_2D_HEXBIN_COLORBAR = _pca_hexbin_traces()
    PCA_2D_DENSITY_TRACE = _pca_density_trace()
    PCA_2D_CASE_TRACES = _pca_case_traces_2d(prepared['pca_2d_x'], prepared['pca_2d_y'])
    
    pca_2d_viewer = go.Figure(PCA_2D_SCATTER_TRACES + PCA_2D_HEXBIN_TRACES + [PCA_2D_HEXBIN_COLORBAR, PCA_2D_DENSITY_TRACE] + PCA_2D_CASE_TRACES)
    scatter_count = len(PCA_2D_SCATTER_TRACES)
    hexbin_count = len(PCA_2D_HEXBIN_TRACES)
    hexbin_colorbar_index = scatter_count + hexbin_count
    density_index = hexbin_colorbar_index + 1
    case_indices = range(density_index + 1, density_index + 1 + len(PCA_2D_CASE_TRACES))
    
    def _pca_mode_mask(mode: str) -> list[bool]:
        visible = [False] * len(pca_2d_viewer.data)
        if mode == "scatter":
            visible[:scatter_count] = [True] * scatter_count
        elif mode == "hexbin":
            visible[scatter_count:hexbin_colorbar_index + 1] = [True] * (hexbin_count + 1)
        elif mode == "density":
            visible[density_index] = True
        else:
            raise ValueError(f"Unknown PCA 2D mode: {mode}")
        for index in case_indices:
            visible[index] = True
        return visible
    
    PCA_2D_MODE_MASKS = {mode: _pca_mode_mask(mode) for mode in ("scatter", "hexbin", "density")}
    for mode, visible in PCA_2D_MODE_MASKS.items():
        if len(visible) != len(pca_2d_viewer.data) or not all(visible[index] for index in case_indices):
            raise ValueError(f"{mode} mode must retain all case overlays.")
    if not PCA_2D_MODE_MASKS["hexbin"][hexbin_colorbar_index]:
        raise ValueError("Hexbin mode must show its count colorbar.")
    if any(PCA_2D_MODE_MASKS[mode][hexbin_colorbar_index] for mode in ("scatter", "density")):
        raise ValueError("The hexbin count colorbar must be hidden outside hexbin mode.")
    
    pca_2d_viewer.update_layout(
        title={"text": f"All retained PCA rows: PC1 vs PC2<br><sup>Explained variance: PC1 {prepared['explained_variance'][0]:.1%} · PC2 {prepared['explained_variance'][1]:.1%}</sup>"},
        template="plotly_white", width=PCA_VIEWER_WIDTH_PX, height=PCA_VIEWER_HEIGHT_PX,
        font={"family": "Arial, sans-serif", "size": 13, "color": "#30343b"},
        xaxis={"title": _pca_axis_title(0), "range": prepared['pca_2d_x_range'], "gridcolor": "#e8ebef", "constrain": "domain"},
        yaxis={"title": _pca_axis_title(1), "range": prepared['pca_2d_y_range'], "gridcolor": "#e8ebef", "scaleanchor": "x", "scaleratio": 1, "constrain": "domain"},
        legend={"orientation": "h", "x": 0, "y": -0.22, "xanchor": "left", "yanchor": "top"},
        updatemenus=[{
            "type": "dropdown", "x": 0.01, "y": 1.08, "xanchor": "left", "yanchor": "top",
            "buttons": [
                {"label": "Transparent Scattergl", "method": "update", "args": [{"visible": PCA_2D_MODE_MASKS["scatter"]}, {}]},
                {"label": "Exact-count hexbins", "method": "update", "args": [{"visible": PCA_2D_MODE_MASKS["hexbin"]}, {}]},
                {"label": "Gaussian density contours", "method": "update", "args": [{"visible": PCA_2D_MODE_MASKS["density"]}, {}]},
            ],
        }, {
            "type": "dropdown", "x": 0.34, "y": 1.08, "xanchor": "left", "yanchor": "top",
            "buttons": [
                {"label": "Full range", "method": "relayout", "args": [{"xaxis.range": prepared['pca_2d_x_range'], "yaxis.range": prepared['pca_2d_y_range']}]},
                {"label": "Central range (1st–99th percentile)", "method": "relayout", "args": [{"xaxis.range": prepared['pca_2d_x_central'], "yaxis.range": prepared['pca_2d_y_central']}]},
            ],
        }],
        margin={"t": 150, "r": 100, "b": 145, "l": 80},
    )
    pca_2d_viewer.update_traces(visible=False)
    for index in list(range(scatter_count)) + list(case_indices):
        pca_2d_viewer.data[index].visible = True
    
    return pca_2d_viewer
    

In [ ]:
pca_2d_viewer = plot_pca_2d(PCA_2D_PREPARED)
for trace, rows in zip(pca_2d_viewer.data[:len(PCA_TYPE_ROW_INDICES)], PCA_TYPE_ROW_INDICES, strict=True):
    assert np.all(PCA_TYPE_IDS[rows] == trace.meta[0])
    assert np.all(PCA_TYPE_NAMES[rows] == trace.meta[1])
    assert np.array_equal(np.asarray(trace.customdata), PCA_DATES_ISO[rows])
    assert np.array_equal(np.asarray(trace.x), PCA_SCORES[rows, 0])
    assert np.array_equal(np.asarray(trace.y), PCA_SCORES[rows, 1])
for trace, case in zip(pca_2d_viewer.data[-PCA_CASE_ROWS.height:], PCA_CASE_ROWS.to_dicts(), strict=True):
    row = int(case["matrix_row"])
    assert trace.customdata[0] == case["endpoint"].isoformat() == PCA_DATES_ISO[row]
    assert int(PCA_TYPE_IDS[row]) == case["type_id"]
    assert (trace.x[0], trace.y[0]) == tuple(PCA_SCORES[row, :2])
PCA_2D_VIEWER_PAYLOAD_BYTES = assert_compact_figure(pca_2d_viewer, PCA_VIEWER_PAYLOAD_BUDGET_BYTES)
pca_2d_viewer

## All-row 3D PCA viewer

Rotate the first three principal components to inspect the retained population and highlighted cases. Full range shows all coordinates; central range zooms each axis to its 1st–99th percentiles without removing rows.

In [ ]:
PCA_3D_PREPARED = {
    "coordinates": PCA_SCORES[:, :3],
    "dates": PCA_DATES_ISO,
    "type_ids": PCA_UNIQUE_TYPE_IDS,
    "labels": PCA_TYPE_LABELS,
    "group_rows": PCA_TYPE_ROW_INDICES,
    "cases": PCA_CASE_ROWS.to_dicts(),
    "row_count": PCA_RETAINED_ROW_COUNT,
    "central_ranges": [_pca_central_range(PCA_SCORES[:, i]) for i in range(3)],
    "explained_variance": pca_model.explained_variance_ratio_[:3],
}

assert sum(len(rows) for rows in PCA_3D_PREPARED["group_rows"]) == PCA_RETAINED_ROW_COUNT
assert np.array_equal(np.sort(np.concatenate(PCA_3D_PREPARED["group_rows"])), np.arange(PCA_RETAINED_ROW_COUNT))
assert PCA_3D_PREPARED["coordinates"].shape == (PCA_RETAINED_ROW_COUNT, 3)
assert np.array_equal(PCA_3D_PREPARED["coordinates"], pca_scores_array[:, :3])
for type_id, label, rows in zip(PCA_3D_PREPARED["type_ids"], PCA_3D_PREPARED["labels"], PCA_3D_PREPARED["group_rows"], strict=True):
    assert np.all(PCA_TYPE_IDS[rows] == type_id)
    assert np.all(PCA_TYPE_NAMES[rows] == label)
    assert np.array_equal(PCA_3D_PREPARED["dates"][rows], PCA_DATES_ISO[rows])
for case in PCA_3D_PREPARED["cases"]:
    row = int(case["matrix_row"])
    assert case["endpoint"].isoformat() == PCA_3D_PREPARED["dates"][row]
    assert int(PCA_TYPE_IDS[row]) == case["type_id"]
    assert PCA_TYPE_NAMES[row] == case["type_name"]
    assert np.array_equal(PCA_3D_PREPARED["coordinates"][row], pca_scores_array[row, :3])

In [ ]:
def plot_pca_3d(prepared):
    PCA_3D_TRACES = []
    for type_id, label, row_indices in zip(prepared["type_ids"], prepared["labels"], prepared["group_rows"], strict=True):
        PCA_3D_TRACES.append(go.Scatter3d(
            x=prepared["coordinates"][row_indices, 0], y=prepared["coordinates"][row_indices, 1], z=prepared["coordinates"][row_indices, 2],
            mode="markers", name=label, meta=[int(type_id), label], customdata=prepared["dates"][row_indices],
            marker={"size": 2.2, "color": "#4c78a8", "opacity": PCA_SCATTER_OPACITY},
            hovertemplate="%{meta[1]} (type %{meta[0]})<br>date=%{customdata}<br>PC1=%{x:.4f}<br>PC2=%{y:.4f}<br>PC3=%{z:.4f}<extra></extra>",
            showlegend=False,
        ))
    if sum(len(trace.x) for trace in PCA_3D_TRACES) != prepared["row_count"]:
        raise ValueError("3D Scatter3d traces must retain every PCA row.")
    
    PCA_3D_CASE_TRACES = []
    for case in prepared["cases"]:
        row = int(case["matrix_row"])
        actual = prepared["coordinates"][row, :3]
        PCA_3D_CASE_TRACES.append(go.Scatter3d(
            x=[actual[0]], y=[actual[1]], z=[actual[2]], mode="markers", name=str(case["short_label"]),
            meta=[str(case["case_id"]), int(case["type_id"]), str(case["type_name"])],
            customdata=[case["endpoint"].isoformat()],
            marker={"size": 6.5, "color": str(case["color"]), "line": {"color": "#111111", "width": 2}},
            hovertemplate="%{meta[0]}<br>%{meta[2]} (type %{meta[1]})<br>date=%{customdata}<br>PC1=%{x:.4f}<br>PC2=%{y:.4f}<br>PC3=%{z:.4f}<extra></extra>",
            showlegend=True,
        ))
    
    pca_3d_viewer = go.Figure(PCA_3D_TRACES + PCA_3D_CASE_TRACES)
    pca_3d_viewer.update_layout(
        title={"text": "All retained PCA rows: rotatable PC1–PC2–PC3 view<br><sup>Explained variance: " + " · ".join(f"PC{i + 1} {prepared['explained_variance'][i]:.1%}" for i in range(3)) + "</sup>"},
        template="plotly_white", width=PCA_VIEWER_WIDTH_PX, height=PCA_VIEWER_HEIGHT_PX,
        font={"family": "Arial, sans-serif", "size": 13, "color": "#30343b"},
        legend={"orientation": "h", "x": 0.5, "y": -0.08, "xanchor": "center", "yanchor": "top"},
        scene={
            "xaxis": {"title": _pca_axis_title(0), "backgroundcolor": "#fafafa", "gridcolor": "#e1e5e9", "showbackground": True},
            "yaxis": {"title": _pca_axis_title(1), "backgroundcolor": "#fafafa", "gridcolor": "#e1e5e9", "showbackground": True},
            "zaxis": {"title": _pca_axis_title(2), "backgroundcolor": "#fafafa", "gridcolor": "#e1e5e9", "showbackground": True},
            "aspectmode": "cube", "camera": {"eye": {"x": 1.4, "y": 1.4, "z": 1.15}},
            "domain": {"x": [0.08, 0.92], "y": [0.05, 0.95]},
        },
        updatemenus=[{
            "type": "dropdown", "x": 0.02, "y": 1.04, "xanchor": "left", "yanchor": "top",
            "buttons": [
                {"label": "Full range", "method": "relayout", "args": [{f"scene.{axis}axis.autorange": True for axis in "xyz"}]},
                {"label": "Central range (1st–99th percentile)", "method": "relayout", "args": [{f"scene.{axis}axis.range": prepared["central_ranges"][i] for i, axis in enumerate("xyz")}]},
            ],
        }],
        margin={"t": 130, "r": 40, "b": 100, "l": 40},
    )
    return pca_3d_viewer

In [ ]:
pca_3d_viewer = plot_pca_3d(PCA_3D_PREPARED)
PCA_3D_VIEWER_PAYLOAD_BYTES = assert_compact_figure(pca_3d_viewer, PCA_VIEWER_PAYLOAD_BUDGET_BYTES)
if PCA_2D_VIEWER_PAYLOAD_BYTES + PCA_3D_VIEWER_PAYLOAD_BYTES > PCA_VIEWER_TOTAL_PAYLOAD_BUDGET_BYTES:
    raise ValueError("Combined saved PCA viewer payload exceeds the explicit collaboration budget.")
pca_3d_viewer

### Item contributions to variation

Compare each item’s share of centered standardized feature variation with its share of complete-case observations. Select a leading component to compare squared centered PCA scores instead; each view ranks the 15 largest contributors, while shares use all prepared items. A large contribution describes influence on variation, not an economic event or a causal effect.

In [ ]:
ITEM_CONTRIBUTION_DISPLAY_LIMIT = 15
item_component_count = min(PCA_COMPONENT_DISPLAY_LIMIT, pca_scores_array.shape[1])
item_contribution_labels = ["Total variation"] + [f"PC {i}" for i in range(1, pca_scores_array.shape[1] + 1)]
item_display_labels = item_contribution_labels[:item_component_count + 1]
centered_scaled = X_scaled - X_scaled.mean(axis=0)
centered_scores = pca_scores_array - pca_scores_array.mean(axis=0)
item_energy = np.column_stack((np.square(centered_scaled).sum(axis=1), np.square(centered_scores)))
energy_totals = item_energy.sum(axis=0)
if not np.isfinite(item_energy).all() or np.any(energy_totals <= 0):
    raise ValueError("Displayed variation must be finite and nonzero.")

item_contribution_metrics = (
    prepared_metadata.select("type_id", "type_name")
    .with_columns([pl.Series(label, item_energy[:, i]) for i, label in enumerate(item_contribution_labels)])
    .group_by("type_id", "type_name")
    .agg(pl.len().alias("observation_count"), *[pl.col(label).sum() for label in item_contribution_labels])
    .with_columns(
        (pl.col("observation_count") / prepared_metadata.height).alias("observation_share"),
        *[(pl.col(label) / energy_totals[i]).alias(label) for i, label in enumerate(item_contribution_labels)],
    )
)
item_contribution_rankings = {
    label: item_contribution_metrics.sort([label, "type_id"], descending=[True, False]).head(ITEM_CONTRIBUTION_DISPLAY_LIMIT)
    for label in item_display_labels
}


In [ ]:
def plot_item_contributions(rankings: dict[str, pl.DataFrame]) -> go.Figure:
    figure = go.Figure()
    labels = list(rankings)
    buttons = []
    for index, label in enumerate(labels):
        ranked = rankings[label]
        item_labels = [f"{name} ({item_id})" for name, item_id in zip(ranked["type_name"], ranked["type_id"])]
        custom = np.column_stack((ranked["observation_count"].to_numpy(), ranked["observation_share"].to_numpy(), ranked[label].to_numpy()))
        for name, values, color in (
            ("Variation share", ranked[label].to_list(), "#0072B2"),
            ("Prepared observation share", ranked["observation_share"].to_list(), "#E69F00"),
        ):
            figure.add_trace(go.Bar(
                x=values, y=item_labels, orientation="h", name=name, marker_color=color,
                visible=index == 0, customdata=custom,
                hovertemplate="%{y}<br>variation share=%{customdata[2]:.4%}<br>observation share=%{customdata[1]:.4%}<br>prepared observations=%{customdata[0]:,.0f}<extra></extra>",
            ))
        visible = [False] * (2 * len(labels))
        visible[2 * index:2 * index + 2] = [True, True]
        buttons.append(dict(label=label, method="update", args=[{"visible": visible}, {"title": f"Item contribution: {label} · top {len(ranked)}"}]))
    figure.update_layout(
        title=f"Item contribution: {labels[0]} · top {len(rankings[labels[0]])}",
        template="plotly_white", width=1100, height=680, barmode="group",
        xaxis_title="Share of all prepared observations or variation", yaxis_title="Item (type ID)",
        font={"size": 12, "color": "#334155"}, margin={"l": 260, "t": 115},
        updatemenus=[dict(buttons=buttons, x=0, y=1.14, xanchor="left", yanchor="top")],
    )
    figure.update_xaxes(tickformat=".0%")
    figure.update_yaxes(autorange="reversed")
    assert_compact_figure(figure)
    return figure


In [ ]:
item_contribution_figure = plot_item_contributions(item_contribution_rankings)
item_contribution_figure

## Temporal PCA behavior explorer

Explore the existing PCA coordinates over time. Dates are descriptor-window endpoints; overlapping windows are not independent events. This is not anomaly detection. Start/end markers identify the first and last displayed observations, not event onset or recovery. The saved explorer runs offline without a kernel connection.

In [ ]:
from IPython.display import HTML
from plotting.temporal_pca import build_temporal_pca_figure, prepare_temporal_pca, temporal_pca_html

temporal_pca_model = prepare_temporal_pca(PCA_3D_PREPARED)

In [ ]:
def plot_temporal_pca(model):
    """Return the offline temporal explorer for the prepared PCA model."""
    return HTML(temporal_pca_html(build_temporal_pca_figure(model), model))

In [ ]:
plot_temporal_pca(temporal_pca_model)